In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("CHANNEL3_API_KEY")

if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

print("Clé chargée :", API_KEY is not None)
print("Longueur de la clé :", len(API_KEY) if API_KEY else 0)

Clé chargée : True
Longueur de la clé : 40


In [8]:
import os
from dotenv import load_dotenv
from channel3_sdk import Channel3

# Charger la clé
load_dotenv()
API_KEY = os.getenv("CHANNEL3_API_KEY")
if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

# Créer le client Channel3 (il s'authentifie avec ta clé)
client = Channel3(api_key=API_KEY)

# Notre première recherche : des sacs en cuir (produit luxe/mode typique)
page = client.products.search(query="leather handbag")

# Combien de produits reçus ?
print("Nombre de produits reçus :", len(page.products))

# Regardons le premier produit
premier = page.products[0]
print("\n--- Premier produit ---")
print("ID :", premier.id)
print("Titre :", premier.title)

Nombre de produits reçus : 20

--- Premier produit ---
ID : rP7Lp38
Titre : Leather Handbag


In [5]:
# Prenons le premier produit et regardons TOUT ce qu'il contient
premier = page.products[0]

print("=== STRUCTURE COMPLÈTE D'UN PRODUIT ===\n")
print("ID :", premier.id)
print("Titre :", premier.title)
print("Description :", premier.description)
print("Marques :", premier.brands)
print("Catégories :", premier.categories)
print("Matériaux :", premier.materials)
print("\n=== LES OFFRES (c'est là qu'est le PRIX) ===")
print(premier.offers)

=== STRUCTURE COMPLÈTE D'UN PRODUIT ===

ID : rP7Lp38
Titre : Leather Handbag
Description : This Bottega Veneta handbag is crafted from smooth, premium natural leather, featuring a compact rectangular shape with rounded corners for an understated elegant design. It has a top handle and an adjustable strap, and is secured with a zipper closure.
Marques : [ProductBrand(id='C2GA', name='Bottega Veneta')]
Catégories : ['Apparel & Accessories > Handbags, Wallets & Cases > Handbags']
Matériaux : ['leather']

=== LES OFFRES (c'est là qu'est le PRIX) ===
[ProductOffer(availability='InStock', domain='jomashop.com', price=Price(currency='USD', price=1745.0, compare_at_price=2425.0), url='https://buy.trychannel3.com/rP7Lp38-IXIa-017wIK?o=01KCA7R6NNSSYTK3M58R46FVE3&aid=01KZHH89F3FX770P3B9RAP6TTH', condition='new', dimensions=None, max_commission_rate=0.0)]


In [4]:
# Le prix est dans offers → price → price
premier = page.products[0]

if premier.offers:                        # s'il y a au moins une offre
    offre = premier.offers[0]             # la première offre
    print("Domaine (retailer) :", offre.domain)
    print("URL :", offre.url)
    print("Prix :", offre.price.price)
    print("Devise :", offre.price.currency)
    print("Prix barré (avant promo) :", offre.price.compare_at_price)
else:
    print("Pas d'offre pour ce produit")

Domaine (retailer) : jomashop.com
URL : https://buy.trychannel3.com/rP7Lp38-IXIa-017wIK?o=01KCA7R6NNSSYTK3M58R46FVE3&aid=01KZHH89F3FX770P3B9RAP6TTH
Prix : 1745.0
Devise : USD
Prix barré (avant promo) : 2425.0
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [9]:
def extraire_infos(produit):
    """Prend un produit Channel3 et renvoie ses infos clés dans un dictionnaire propre."""

    # La marque : c'est une liste, on prend la première si elle existe
    marque = produit.brands[0].name if produit.brands else None

    # La catégorie principale
    categorie = produit.category.title if produit.category else None

    # Le prix : niché dans la première offre
    if produit.offers:
        offre = produit.offers[0]
        prix = offre.price.price
        prix_barre = offre.price.compare_at_price
        devise = offre.price.currency
        retailer = offre.domain
    else:
        prix = prix_barre = devise = retailer = None

    return {
        "id": produit.id,
        "titre": produit.title,
        "marque": marque,
        "categorie": categorie,
        "prix": prix,
        "prix_barre": prix_barre,
        "devise": devise,
        "retailer": retailer
    }

# Testons sur le premier produit
infos = extraire_infos(page.products[0])
print(infos)

{'id': 'rP7Lp38', 'titre': 'Leather Handbag', 'marque': 'Bottega Veneta', 'categorie': 'Handbags', 'prix': 1745.0, 'prix_barre': 2425.0, 'devise': 'USD', 'retailer': 'jomashop.com'}


In [10]:
import pandas as pd

# Appliquer notre fonction à TOUS les produits reçus
donnees = [extraire_infos(produit) for produit in page.products]

# Transformer la liste de dictionnaires en tableau pandas
df = pd.DataFrame(donnees)

# Regarder le résultat
print("Nombre de lignes :", len(df))
df.head(10)

Nombre de lignes : 20


,id,titre,marque,categorie,prix,prix_barre,devise,retailer
0,rP7Lp38,Leather Handbag,Bottega Veneta,Handbags,1745.00,2425.00,USD,jomashop.com
1,gW1gxYV,Prada Darling Leather Handbag,Prada,"Handbags, Wallets & Cases",3550.00,NaN,USD,prada.com
2,t2JZlpl,Amazona 23 Leather Handbag,LOEWE,Handbags,2774.00,4170.00,USD,luosophy.com
3,oqE5xBI,Vita Leather Handbag,Golden Goose,Handbags,724.00,1278.00,USD,us.thahab.com
4,om5mRqD,Leather Handbag,Bottega Veneta,Handbags,4011.33,6171.27,USD,netdressed.com
5,JAOAQX5,Darling leather handbag,Prada,"Handbags, Wallets & Cases",3550.00,NaN,USD,prada.com
6,abH9Of0,Classic Genuine Leather Handbag,Nordstrom,Handbags,228.00,NaN,USD,nordstrom.com
7,UJIGnIj,Leather handbag,Mango,"Handbags, Wallets & Cases",269.99,NaN,USD,shop.mango.com
8,9CI1FIK,Medium Leather Handbag,Longchamp,Handbags,449.97,830.00,USD,nordstromrack.com
9,FJqwpa2,Braided Leather Handbag,Nordstrom,Handbags,199.00,NaN,USD,nordstrom.com


In [11]:
# Les catégories qu'on veut analyser (produits mode/luxe)
requetes = [
    "leather handbag",
    "leather shoes",
    "luxury watch",
    "wool coat",
    "silk scarf",
    "sunglasses"
]

# On va collecter tous les produits ici
tous_les_produits = []

for requete in requetes:
    print(f"Recherche : {requete}...")
    page = client.products.search(query=requete)

    for produit in page.products:
        infos = extraire_infos(produit)
        infos["requete"] = requete   # on garde trace de la recherche d'origine
        tous_les_produits.append(infos)

    print(f"  → {len(page.products)} produits récupérés")

# Transformer le tout en un grand DataFrame
df = pd.DataFrame(tous_les_produits)
print("\n=== TOTAL ===")
print("Nombre total de produits :", len(df))
df.head()

Recherche : leather handbag...
  → 20 produits récupérés
Recherche : leather shoes...
  → 20 produits récupérés
Recherche : luxury watch...
  → 20 produits récupérés
Recherche : wool coat...
  → 20 produits récupérés
Recherche : silk scarf...
  → 20 produits récupérés
Recherche : sunglasses...
  → 20 produits récupérés

=== TOTAL ===
Nombre total de produits : 120


,id,titre,marque,categorie,prix,prix_barre,devise,retailer,requete
0,rP7Lp38,Leather Handbag,Bottega Veneta,Handbags,1745.0,2425.0,USD,jomashop.com,leather handbag
1,gW1gxYV,Prada Darling Leather Handbag,Prada,"Handbags, Wallets & Cases",3550.0,NaN,USD,prada.com,leather handbag
2,t2JZlpl,Amazona 23 Leather Handbag,LOEWE,Handbags,2774.0,4170.0,USD,luosophy.com,leather handbag
3,oqE5xBI,Vita Leather Handbag,Golden Goose,Handbags,724.0,1278.0,USD,us.thahab.com,leather handbag
4,BNLviZP,Arezzo Grained Leather Handbag,ATP Atelier,Handbags,590.0,NaN,USD,us.atpatelier.com,leather handbag


In [12]:
# --- D1 : PROFILING — état des lieux de la qualité ---
print("=== DIAGNOSTIC QUALITÉ ===\n")
print("Dimensions du tableau :", df.shape)          # (lignes, colonnes)
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())                            # combien de NaN par colonne
print("\nDoublons (même id) :", df.duplicated(subset=["id"]).sum())
print("\nDevises présentes :", df["devise"].unique())

=== DIAGNOSTIC QUALITÉ ===

Dimensions du tableau : (120, 9)

Valeurs manquantes par colonne :
id             0
titre          0
marque         0
categorie      0
prix           0
prix_barre    73
devise         0
retailer       0
requete        0
dtype: int64

Doublons (même id) : 0

Devises présentes : <StringArray>
['USD']
Length: 1, dtype: str


In [14]:
# --- D2 : NETTOYAGE ---
# On part d'une copie propre
df_clean = df.copy()

# 1. Enlever les produits sans prix (inexploitables pour une analyse de prix)
df_clean = df_clean[df_clean["prix"].notna()]

# 2. Enlever les doublons éventuels (même id)
df_clean = df_clean.drop_duplicates(subset=["id"])

# 3. Ne garder que les prix en USD (pour comparer ce qui est comparable)
df_clean = df_clean[df_clean["devise"] == "USD"]

# 4. Créer une colonne "remise" quand il y a un prix barré
df_clean["remise_pct"] = (
    (df_clean["prix_barre"] - df_clean["prix"]) / df_clean["prix_barre"] * 100
).round(1)

print("Avant nettoyage :", len(df), "produits")
print("Après nettoyage :", len(df_clean), "produits")
df_clean.head()

Avant nettoyage : 120 produits
Après nettoyage : 120 produits


,id,titre,marque,categorie,prix,prix_barre,devise,retailer,requete,remise_pct
0,rP7Lp38,Leather Handbag,Bottega Veneta,Handbags,1745.0,2425.0,USD,jomashop.com,leather handbag,28.0
1,gW1gxYV,Prada Darling Leather Handbag,Prada,"Handbags, Wallets & Cases",3550.0,NaN,USD,prada.com,leather handbag,NaN
2,t2JZlpl,Amazona 23 Leather Handbag,LOEWE,Handbags,2774.0,4170.0,USD,luosophy.com,leather handbag,33.5
3,oqE5xBI,Vita Leather Handbag,Golden Goose,Handbags,724.0,1278.0,USD,us.thahab.com,leather handbag,43.3
4,BNLviZP,Arezzo Grained Leather Handbag,ATP Atelier,Handbags,590.0,NaN,USD,us.atpatelier.com,leather handbag,NaN


In [15]:
# --- E1 : Positionnement prix par marque ---
analyse_marque = df_clean.groupby("marque").agg(
    nb_produits=("id", "count"),
    prix_moyen=("prix", "mean"),
    prix_min=("prix", "min"),
    prix_max=("prix", "max")
).round(0).sort_values("prix_moyen", ascending=False)

print("=== POSITIONNEMENT PRIX PAR MARQUE ===")
analyse_marque

=== POSITIONNEMENT PRIX PAR MARQUE ===


,nb_produits,prix_moyen,prix_min,prix_max
marque,,,,
Louis Vuitton,2,38210.0,421.0,76000.0
Audemars Piguet,2,23775.0,20155.0,27395.0
Chopard,2,22952.0,16950.0,28955.0
Bell & Ross,1,22770.0,22770.0,22770.0
Rolex,2,16730.0,12950.0,20511.0
...,...,...,...,...
Reebok,3,69.0,47.0,85.0
Polaroid,2,68.0,62.0,74.0
Le Specs,2,67.0,49.0,85.0


In [16]:
# --- E2 : Positionnement par catégorie de recherche ---
analyse_categorie = df_clean.groupby("requete").agg(
    nb_produits=("id", "count"),
    prix_moyen=("prix", "mean"),
    prix_median=("prix", "median"),
    prix_min=("prix", "min"),
    prix_max=("prix", "max")
).round(0).sort_values("prix_moyen", ascending=False)

print("=== ÉCARTS DE PRIX PAR CATÉGORIE ===")
analyse_categorie

=== ÉCARTS DE PRIX PAR CATÉGORIE ===


,nb_produits,prix_moyen,prix_median,prix_min,prix_max
requete,,,,,
luxury watch,20,13075.0,4972.0,169.0,76000.0
leather handbag,20,2013.0,1069.0,199.0,5040.0
wool coat,20,1330.0,813.0,112.0,6330.0
leather shoes,20,332.0,133.0,18.0,1718.0
sunglasses,20,233.0,202.0,49.0,619.0
silk scarf,20,230.0,185.0,49.0,665.0


In [17]:
# Sauvegarder le tableau nettoyé dans data/ (pour Power BI ensuite)
df_clean.to_csv("../data/produits_clean.csv", index=False)
print("Fichier sauvegardé : data/produits_clean.csv")
print("Prêt pour Power BI !")

Fichier sauvegardé : data/produits_clean.csv
Prêt pour Power BI !
